## 1.0 Output upscaled parameters for flac input

In [59]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------
#  User settings
# ----------------------------------------
csv_folder = r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\predictions_12135"


sigma3     = 2000.0
Ea         = np.linspace(0, 0.132, 65)


#  Helper: slope calculator
# ----------------------------------------
def calc_slopes(x, y):
    slopes = np.zeros(len(x))
    for i in range(len(x)):
        if i == 0:
            slopes[i] = (y[1] - y[0]) / (x[1] - x[0])
        elif i == len(x) - 1:
            slopes[i] = (y[-1] - y[-2]) / (x[-1] - x[-2])
        else:
            slopes[i] = (y[i+1] - y[i-1]) / (x[i+1] - x[i-1])
    return slopes

# ----------------------------------------
#  Storage containers
# ----------------------------------------
results          = []
dil_ang_lists    = []
dil_ps_lists     = []
fric_ang_lists   = []
fric_ps_lists    = []
coh_lists        = []
coh_ps_lists     = []

# ----------------------------------------
#  Process each prediction CSV
# ----------------------------------------
for fname in sorted(os.listdir(csv_folder)):
    if not fname.endswith('.csv'):
        continue
    
    df    = pd.read_csv(os.path.join(csv_folder, fname))
    plast = df['plastic_strain'].values
    vol   = df['volumetric_strain'].values
    q     = df['soil_strength'].values

    # initial pore pressure
    p1 = df['pore_pressure'].iloc[1]

    # 1) friction angle with known cohesion c0 = 0 kPa
    sigma1  = q + sigma3
    phi_all   = (np.degrees(np.arctan(np.sqrt(sigma1 / sigma3))) - 45.0) * 2.0
    phi_all = np.clip(phi_all, 0.0, None)

    # 2) dilation angle
    slopes = calc_slopes(Ea, vol)
    ang_d  = np.degrees(np.arcsin(slopes / (slopes + 2)))

    # plastic region mask
    mask_soft_0   = plast > 0
    mask_soft_6e4   = plast > 0.0003
    mask_soft   = plast > 0.0003
    
    plast_soft  = plast[mask_soft]
    plast_soft_dil=plast[mask_soft_dil]

    phi_soft = phi_all[mask_soft]
    #psi_soft = ang_d[mask_soft]
    psi_soft = ang_d[mask_soft_6e4]
    
    # ---- cohesion (computed on the same mask_soft) ----
    phi_rad = np.radians(phi_all)
    sin_phi = np.sin(phi_rad)
    cos_phi = np.cos(phi_rad)

    T1 = 1 + sin_phi / (1 - sin_phi)
    T2 = cos_phi / (1 - sin_phi)

    c_all = (sigma1 - sigma3 * T1) / (2 * T2)

    c_soft = c_all[mask_soft]
    c_clip = np.maximum(c_soft, 0.0)

    # ---- find start index for "valid" tables ----
    c_tol   = 1e-6   # kPa, adjust if needed
    phi_tol = 0.1    # degrees, adjust if needed

    idx_c   = np.where(c_clip > c_tol)[0]
    idx_phi = np.where(phi_soft > phi_tol)[0]

    start_c   = idx_c[0]   if len(idx_c)   else None
    start_phi = idx_phi[0] if len(idx_phi) else None

    # choose a common start so all parameters are valid
    starts = [s for s in [start_c, start_phi] if s is not None]
    start = max(starts) if starts else None

    if start is None:
        plast_final = np.array([], dtype=float)
        plast_final_dil = np.array([], dtype=float)
        phi_final   = np.array([], dtype=float)
        psi_final   = np.array([], dtype=float)
        coh_final   = np.array([], dtype=float)
    else:
        plast_final = plast_soft[start:]
        plast_final_dil = plast_soft_dil[start:]
        phi_final   = phi_soft[start:]
        psi_final   = psi_soft[start:]
        coh_final   = c_clip[start:]

    # collect for softening tables (all aligned)
    fric_ps_lists.append(plast_final)
    fric_ang_lists.append(phi_final)

    dil_ps_lists.append(plast_final_dil)
    dil_ang_lists.append(psi_final)

    coh_ps_lists.append(plast_final)
    coh_lists.append(coh_final)

    
    # 5) pick peak values
    phi_peak = np.nanmax(phi_all[mask_soft])
    psi_peak = np.nanmax(ang_d[mask_soft])
    c_peak   = np.nanmax(c_all[mask_soft])

 
    # 6) compute E50 as a tangent modulus at 50% of peak stress
    i_peak = np.nanargmax(q)
    q_peak = q[i_peak]
    q_half = 0.5 * q_peak

    # find the index closest to 50% of peak (including the peak itself)
    half_idx = np.abs(q[:i_peak+1] - q_half).argmin()

    # compute pointwise slopes (dq/dEa) up to the peak
    slopes = np.gradient(q[:i_peak+1], Ea[:i_peak+1])

    # set up a window of ±2 points around the half-peak
    win = 2
    lo = max(0, half_idx - win)
    hi = min(i_peak, half_idx + win)

    # if we have at least 3 points in [lo..hi], do a small-window linear fit;
    # otherwise fallback to the single-point finite-difference slope
    if hi - lo >= 2:
        E50, _, _, _, _ = stats.linregress(Ea[lo:hi+1], q[lo:hi+1])
    else:
        E50 = slopes[half_idx]

    # 7) estimate Poisson's ratio ν from volumetric strain slope
    slopes_v = calc_slopes(Ea, vol)
    min_vol_index = np.argmin(vol)
    poisson_ratio_end_index = max(2, min_vol_index)
    slopes_segment = slopes_v[: poisson_ratio_end_index + 1]
    i_max_mag = np.nanargmax(np.abs(slopes_segment))
    half_window = 3
    lo = max(0, i_max_mag - half_window)
    hi = min(poisson_ratio_end_index, i_max_mag + half_window)
    if (hi - lo) >= 2:
        slope_fit, _, _, _, _ = stats.linregress(Ea[lo : hi + 1], vol[lo : hi + 1])
    else:
        slope_fit = slopes_v[i_max_mag]
    slope_abs = abs(slope_fit)
    nu_raw = (1.0 - slope_abs) / 2.0
    nu = float(np.clip(nu_raw, 0.0, 0.5))
    #nu = (1.0 - max_slope) / 2.0



    # 8) bulk & shear moduli
    if not np.isnan(nu):
        K = E50 / (3 * (1 - 2*nu))
        G = E50 / (2 * (1 + nu))
    else:
        K = np.nan
        G = np.nan

    # 9) store results
    results.append({
        'filename':        fname,
        'initial_pore_pp': p1,
        'E50':             E50,
        'nu':              nu,
        'K':               K,
        'G':               G,
        'phi_peak_deg':    phi_peak,
        'psi_peak_deg':    psi_peak,
        'c_peak':          c_peak,
    })

# ----------------------------------------
#  Write flat results CSV
# ----------------------------------------
out_file1 = 'input_parameters_upscale_peak_12135.csv'
if os.path.exists(out_file1):
    os.remove(out_file1)
pd.DataFrame(results).to_csv(out_file1, index=False)
print(f"Wrote {out_file1} with {len(results)} rows.")

out_file = 'input_parameters_upscale_peak_12135.dat'
if os.path.exists(out_file):
    os.remove(out_file)
pd.DataFrame(results).to_csv(out_file, index=False)
print(f"Wrote {out_file} with {len(results)} rows.")
# ----------------------------------------
#  Pivot into direction‐sample format
# ----------------------------------------
df_res = pd.DataFrame(results)
N      = len(df_res)
tags   = np.empty(N, dtype='U1')
tags[:125]    = 'Z'
tags[125:250] = 'X'
tags[250:375] = 'Y'
df_res['direction'] = tags
df_res['sample']    = np.arange(N) % 125

pivot = df_res.pivot(
    index='sample',
    columns='direction',
    values=[
        'initial_pore_pp','E50','nu','K','G',
        'phi_peak_deg','psi_peak_deg','c_peak'
    ]
)
pivot.columns = [f"{metric}_{dir}" for metric, dir in pivot.columns]
pivot = pivot.reset_index()

out_file2 = 'input_parameters_upscale_combined_peak_12135_mask.dat'
if os.path.exists(out_file2):
    os.remove(out_file2)
pivot.to_csv(out_file2, index=False,header=False,)
print(f"Wrote {out_file2} with {pivot.shape[0]} rows.")
out_file3 = 'input_parameters_upscale_combined_peak_12135_mask.csv'
if os.path.exists(out_file3):
    os.remove(out_file3)
pivot.to_csv(out_file3, index=False)
print(f"Wrote {out_file3} with {pivot.shape[0]} rows.")
# ----------------------------------------
#  Build strain‐softening tables outputs
# ----------------------------------------
def pad_series(arrs, maxlen):
    return np.array([
        np.pad(a, (0, maxlen - len(a)), constant_values=np.nan)
        for a in arrs
    ])

dil_ang_p  = pad_series(dil_ang_lists,  max(len(a) for a in dil_ang_lists))
dil_ps_p   = pad_series(dil_ps_lists,   max(len(a) for a in dil_ps_lists))
fric_ang_p = pad_series(fric_ang_lists, max(len(a) for a in fric_ang_lists))
fric_ps_p  = pad_series(fric_ps_lists,  max(len(a) for a in fric_ps_lists))
coh_p      = pad_series(coh_lists,      max(len(a) for a in coh_lists))
coh_ps_p   = pad_series(coh_ps_lists,   max(len(a) for a in coh_ps_lists))

output_files = [
    'all_dilation_plastic_strains_12135_mask.csv',
    'all_dilation_angles_12135_mask.csv',
    'all_friction_plastic_strains_12135_mask.csv',
    'all_friction_angles_12135_mask.csv',
    'all_cohesion_plastic_strains_12135_mask.csv',
    'all_cohesion_12135_mask.csv',
]
for fn in output_files:
    if os.path.exists(fn):
        os.remove(fn)

pd.DataFrame(dil_ps_p)  .to_csv('all_dilation_plastic_strains_12135_mask.csv',    index=False, header=False)
pd.DataFrame(dil_ang_p) .to_csv('all_dilation_angles_12135_mask.csv',             index=False, header=False)
pd.DataFrame(fric_ps_p) .to_csv('all_friction_plastic_strains_12135_mask.csv',    index=False, header=False)
pd.DataFrame(fric_ang_p).to_csv('all_friction_angles_12135_mask.csv',             index=False, header=False)
pd.DataFrame(coh_ps_p)  .to_csv('all_cohesion_plastic_strains_12135_mask.csv',    index=False, header=False)
pd.DataFrame(coh_p)     .to_csv('all_cohesion_12135_mask.csv',                    index=False, header=False)

print("Written strain‐softening tables:")
print(f" • dilation (plast):    {dil_ps_p.shape}")
print(f" • dilation (angle):    {dil_ang_p.shape}")
print(f" • friction (plast):    {fric_ps_p.shape}")
print(f" • friction (angle):    {fric_ang_p.shape}")
print(f" • cohesion (plast):    {coh_ps_p.shape}")
print(f" • cohesion (value):    {coh_p.shape}")


Wrote input_parameters_upscale_peak_12135.csv with 375 rows.
Wrote input_parameters_upscale_peak_12135.dat with 375 rows.
Wrote input_parameters_upscale_combined_peak_12135_mask.dat with 125 rows.
Wrote input_parameters_upscale_combined_peak_12135_mask.csv with 125 rows.
Written strain‐softening tables:
 • dilation (plast):    (375, 62)
 • dilation (angle):    (375, 58)
 • friction (plast):    (375, 58)
 • friction (angle):    (375, 58)
 • cohesion (plast):    (375, 58)
 • cohesion (value):    (375, 58)


In [60]:
import os
import re
import numpy as np
import pandas as pd
# — USER SETTINGS —
combined_path = 'dilation_tables_40.dat'
output_files = [
    combined_path,
    'dilz_tables_12135_mask.dat',
    'dilx_tables_12135_mask.dat',
    'dily_tables_12135_mask.dat',
]

# 0) Remove old files if they exist
for f in output_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed old file → {f}")

# 1) Load the CSVs you generated
dil_mask  = pd.read_csv('all_dilation_plastic_strains_12135_mask.csv', header=None).values
dil_ang = pd.read_csv('all_dilation_angles_12135_mask.csv',          header=None).values

# 2) Build up to 40 pairs per sample, write combined file
combined_path = 'dilation_tables_40.txt'
with open(combined_path, 'w') as out:
    for idx, (mask_row, ang_row) in enumerate(zip(dil_mask, dil_ang), start=0):
        pairs = []
        for p, a in zip(mask_row, ang_row):
            if np.isnan(p) or np.isnan(a):
                break
            pairs.append(f"({p:g},{a:g})")
            if len(pairs) >= 50:
                break
        out.write(f"table 'dil_{idx}' add " + ''.join(pairs) + '\n')
print(f"Wrote combined 375 lines with ≤40 pairs each → {combined_path}")

# 3) Read it back and split into 3×125 blocks
with open(combined_path) as f:
    lines = [L.strip() for L in f if L.strip()]
if len(lines) != 375:
    raise RuntimeError(f"Expected 375 lines, got {len(lines)}")

n = 125
blocks = {
    'dilz': lines[   0:   n],
    'dilx': lines[  n: 2*n],
    'dily': lines[2*n: 3*n],
}

# 4) Rename and write per‐direction files
for prefix, block in blocks.items():
    out_lines = []
    for i, line in enumerate(block, start=1):
        renamed = re.sub(r"table\s+'dil_\d+'",
                         f"table '{prefix}_{i}'",
                         line)
        out_lines.append(renamed)

    fname = f"{prefix}_table_12135_mask.dat"
    with open(fname, 'w') as f:
        f.write('\n'.join(out_lines))
    print(f"Wrote {len(out_lines)} lines → {fname}")
    import os
import re
import numpy as np
import pandas as pd

# — USER SETTINGS —
combined_path = 'friction_tables_40.dat'
output_files = [
    combined_path,
    'fricz_table_12135_mask.dat',
    'fricx_tables_12135_mask.dat',
    'fricy_tables_12135_mask.dat',
]

# 0) Remove old files if they exist
for f in output_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed old file → {f}")

# 1) Load your two friction CSVs
fric_mask  = pd.read_csv('all_friction_plastic_strains_12135_mask.csv', header=None).values
fric_ang = pd.read_csv('all_friction_angles_12135_mask.csv',          header=None).values

# 2) Build up to 40 (plastic_strain,friction_angle) pairs per sample,
#    write to a combined file
combined_path = 'friction_tables_40.txt'
with open(combined_path, 'w') as out:
    for idx, (mask_row, ang_row) in enumerate(zip(fric_mask, fric_ang), start=0):
        pairs = []
        for p, a in zip(mask_row, ang_row):
            if np.isnan(p) or np.isnan(a):
                break
            pairs.append(f"({p:g},{a:g})")
            if len(pairs) >= 50:
                break
        out.write(f"table 'fric_{idx}' add " + ''.join(pairs) + '\n')
print(f"Wrote combined 375 lines with ≤40 pairs each → {combined_path}")

# 3) Read back and split into 3×125 blocks for Z, X, Y
with open(combined_path) as f:
    lines = [L.strip() for L in f if L.strip()]
if len(lines) != 375:
    raise RuntimeError(f"Expected 375 lines, got {len(lines)}")

n = 125
blocks = {
    'fricz': lines[   0:   n],
    'fricx': lines[  n: 2*n],
    'fricy': lines[2*n: 3*n],
}

# 4) Rename and write per‐direction files
for prefix, block in blocks.items():
    out_lines = []
    for i, line in enumerate(block, start=1):
        renamed = re.sub(
            r"table\s+'fric_\d+'",
            f"table '{prefix}_{i}'",
            line
        )
        out_lines.append(renamed)

    fname = f"{prefix}_table_12135_mask.dat"
    with open(fname, 'w') as f:
        f.write('\n'.join(out_lines))
    print(f"Wrote {len(out_lines)} lines → {fname}")

import os
import re
import numpy as np
import pandas as pd

# — USER SETTINGS —
combined_path = 'cohesion_tables_40.dat'
output_files = [
    combined_path,
    'cohz_table_12135_mask.dat',
    'cohx_table_12135_mask.dat',
    'cohy_table_12135_mask.dat',
]

# 0) Remove old files if they exist
for f in output_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed old file → {f}")

# 1) Load your cohesion CSVs
coh_mask = pd.read_csv('all_cohesion_plastic_strains_12135_mask.csv', header=None).values
coh    = pd.read_csv('all_cohesion_12135_mask.csv',                  header=None).values

# 2) Build up to 40 (plastic_strain, cohesion) pairs per sample
with open(combined_path, 'w') as out:
    for idx, (mask_row, c_row) in enumerate(zip(coh_mask, coh), start=0):
        pairs = []
        for p, c in zip(mask_row, c_row):
            if np.isnan(p) or np.isnan(c):
                break
            pairs.append(f"({p:g},{c:g})")
            if len(pairs) >= 50:
                break
        out.write(f"table 'coh_{idx}' add " + ''.join(pairs) + '\n')
print(f"Wrote combined {len(coh_mask)} lines → {combined_path}")

# 3) Read back and split into 3×N blocks for Z, X, Y
with open(combined_path) as f:
    lines = [L.strip() for L in f if L.strip()]
if len(lines) % 3 != 0:
    raise RuntimeError(f"Expected number of lines to be a multiple of 3, got {len(lines)}")

n = len(lines) // 3
blocks = {
    'cohz': lines[   0:   n],
    'cohx': lines[  n: 2*n],
    'cohy': lines[2*n: 3*n],
}

# 4) Rename and write per-direction files
for prefix, block in blocks.items():
    fname = f"{prefix}_table_12135_mask.dat"
    with open(fname, 'w') as f:
        for i, line in enumerate(block, start=1):
            renamed = re.sub(
                r"table\s+'coh_\d+'",
                f"table '{prefix}_{i}'",
                line
            )
            f.write(renamed + '\n')
    print(f"Wrote {len(block)} lines → {fname}")


Wrote combined 375 lines with ≤40 pairs each → dilation_tables_40.txt
Wrote 125 lines → dilz_table_12135_mask.dat
Wrote 125 lines → dilx_table_12135_mask.dat
Wrote 125 lines → dily_table_12135_mask.dat
Removed old file → fricz_table_12135_mask.dat
Wrote combined 375 lines with ≤40 pairs each → friction_tables_40.txt
Wrote 125 lines → fricz_table_12135_mask.dat
Wrote 125 lines → fricx_table_12135_mask.dat
Wrote 125 lines → fricy_table_12135_mask.dat
Removed old file → cohesion_tables_40.dat
Removed old file → cohz_table_12135_mask.dat
Removed old file → cohx_table_12135_mask.dat
Removed old file → cohy_table_12135_mask.dat
Wrote combined 375 lines → cohesion_tables_40.dat
Wrote 125 lines → cohz_table_12135_mask.dat
Wrote 125 lines → cohx_table_12135_mask.dat
Wrote 125 lines → cohy_table_12135_mask.dat
